# Notebook 3 — Limpieza de Datos 🧹
**TFG: Detección de Intrusiones a Gran Escala utilizando ML Distribuido y Big Data**

**Autor:** Eduardo Morillas Rodríguez

**Entrada:** Parquet de `DATA_PARQUET_PATH` (NB1) · **Salida:** `DATA_CLEAN_PATH`

---

### Hallazgos que motivan esta limpieza

| Fuente | Hallazgo | Acción |
|--------|----------|--------|
| NB1 (In[6]) | `Flow ID`, `Src IP`, `Dst IP`, `Src Port` con 51% nulls | Eliminar (identificadores, no features) |
| NB1 (In[8]) | 131.799 infinitos en `Flow Byts/s` y `Flow Pkts/s` | Reemplazar por null → dropna |
| NB1 (In[7]) | 518.436 duplicados exactos (3,19%) | Eliminar duplicados |
| NB1 (In[5]) | `Flow Duration` min = −919 mil millones | Clipear negativos a 0 |
| NB2 (temporal) | 14 timestamps con año < 2000 (epoch 1970) | Filtrar |
| EDA (vis 05, 07, 30) | Distribuciones distorsionadas por outliers extremos | Confirman inf/negativos |

### Respaldo en la literatura

- **Göcs & Johanyák (2023)**: Eliminan columnas no-feature (Timestamp, IDs) primero,
  luego filas con inf y negativos. De 80 → 69 columnas.
- **ml-ids (cstub/ml-ids)**: Reemplazan inf → NaN, negativos → NaN
  **excepto** `Init Fwd Win Byts` y `Init Bwd Win Byts` (centinela -1).
- **Songma et al. (2023)**: Data cleaning → EDA → normalización.
- **Bouidaine et al. (2025)**: Duplicate removal, missing values, encoding, scaling.

### Qué NO se hace en este notebook (y por qué)

| Operación | ¿Dónde? | Motivo |
|-----------|---------|--------|
| Label encoding | NB5 | NB4 necesita `Label` como string para agrupar/visualizar |
| VectorAssembler | NB5 | NB4 va a añadir/eliminar features → el vector cambiaría |
| StandardScaler | NB5 | Debe hacerse tras train/test split para evitar data leakage |
| Drop de `Timestamp` | NB4 | Se conserva para extraer features temporales (hour, day_of_week) |

---
## Imports y Configuración

In [1]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, FloatType, IntegerType, LongType
from config import *

spark = get_spark_session("TFG_IDS_NB3_Limpieza")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/04 21:12:09 WARN Utils: Your hostname, eespcachcpro01, resolves to a loopback address: 127.0.1.1; using 10.151.52.2 instead (on interface ens3f0)
26/05/04 21:12:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/04 21:12:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/04 21:12:10 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).


In [2]:
# === CARGAR DATOS ===
df = spark.read.parquet(DATA_PARQUET_PATH)
initial_rows = df.count()
initial_cols = len(df.columns)
print(f"📊 Dataset cargado: {initial_rows:,} filas × {initial_cols} columnas")
print(f"\nColumnas presentes:\n{df.columns}")

[Stage 1:========================================>              (92 + 32) / 124]

📊 Dataset cargado: 16,232,943 filas × 84 columnas

Columnas presentes:
['Dst Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max', 'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean', 'Bwd Pkt Len Std', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Len', 'Bwd Header Len', 'Fwd Pkts/s', 'Bwd Pkts/s', 'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean', 'Pkt Len Std', 'Pkt Len Var', 'FIN Flag Cnt', 'SYN Flag Cnt', 'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt', 'CWE Flag Count', 'ECE Flag Cnt', 'Down/Up Ratio', 'Pkt Size Avg', 'Fwd Seg Size Avg', 'Bwd Seg Siz

---
## 3.1 — Eliminación de Columnas de Identificación

Las columnas `Flow ID`, `Src IP`, `Dst IP` y `Src Port` son **identificadores de red**,
no features estadísticas de flujo. Incluirlas causaría:
- **Data leakage** (el modelo memorizaría IPs/puertos específicos del lab).
- **Alta cardinalidad** (miles de valores únicos → OHE inviable).
- **51% de nulls** (solo existían en 1 de 10 CSVs).

> **Ref:** Göcs & Johanyák (2023) eliminan columnas no-feature del CSE-CIC-IDS 2018
> antes de cualquier análisis, incluyendo identificadores de flujo.

**⚠️ IMPORTANTE:** Este paso DEBE ejecutarse ANTES de `dropna()`, ya que
los nulls en estas columnas (51%) eliminarían millones de filas válidas.

**Nota:** `Timestamp` se conserva para extraer features temporales en NB4.

In [3]:
print("🔍 Eliminando columnas de identificación...")

ID_COLS = ["Flow ID", "Src IP", "Dst IP", "Src Port"]

existing_id_cols = [c for c in ID_COLS if c in df.columns]

if existing_id_cols:
    for c in existing_id_cols:
        n_nulls = df.filter(F.col(c).isNull()).count()
        n_distinct = df.filter(F.col(c).isNotNull()).select(c).distinct().count()
        print(f"  · {c}: {n_nulls:,} nulls, {n_distinct:,} valores únicos")

    df = df.drop(*existing_id_cols)
    print(f"\n  ✅ Eliminadas {len(existing_id_cols)} columnas: {existing_id_cols}")
else:
    print("  ✅ No se encontraron columnas de identificación")

print(f"  Columnas restantes: {len(df.columns)}")

🔍 Eliminando columnas de identificación...


  · Flow ID: 8,284,195 nulls, 5,030,830 valores únicos


  · Src IP: 8,284,195 nulls, 31,291 valores únicos


  · Dst IP: 8,284,195 nulls, 27,076 valores únicos


[Stage 34:==================================================>   (116 + 8) / 124]

  · Src Port: 8,284,195 nulls, 64,898 valores únicos

  ✅ Eliminadas 4 columnas: ['Flow ID', 'Src IP', 'Dst IP', 'Src Port']
  Columnas restantes: 80


---
## 3.2 — Eliminación de Duplicados Exactos

**Evidencia:** NB1 detectó 518.436 registros duplicados (3,19%).
Los duplicados inflan artificialmente el peso de ciertos patrones.

> **Ref:** Bouidaine et al. (2025), Suresh Kumar (2025) incluyen
> duplicate removal como primer paso de limpieza.

In [4]:
print("🔍 Eliminando duplicados exactos...")
rows_before = df.count()
df = df.dropDuplicates()
rows_after = df.count()

print(f"  Filas antes:  	{rows_before:,}")
print(f"  Filas eliminadas: {rows_before - rows_after:,} ({(rows_before - rows_after) / rows_before * 100:.2f}%)")
print(f"  Filas restantes:  {rows_after:,}")

🔍 Eliminando duplicados exactos...


26/05/04 21:12:38 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 45:=====================================>                (89 + 39) / 128]

  Filas antes:  	16,232,943
  Filas eliminadas: 433,195 (2.67%)
  Filas restantes:  15,799,748


---
## 3.3 — Tratamiento de Valores Infinitos y NaN

**Evidencia:** NB1 (In[8]) detectó 95.760 infinitos en `Flow Pkts/s` y 36.039 en `Flow Byts/s`,
producidos por divisiones por cero (flujos de duración 0).
EDA (vis 05a, 05b, 07, 30) confirma distribuciones distorsionadas.

**Estrategia:** Reemplazar `inf`, `-inf` y `NaN` por `null`, luego eliminar filas afectadas.

> **Ref:** Göcs & Johanyák (2023) eliminan filas con inf/-inf.
> ml-ids (cstub) reemplaza inf → NaN y luego imputa.

In [5]:
print("🔍 Reemplazando infinitos y NaN por null...")

numeric_cols = [
    f.name for f in df.schema.fields
    if isinstance(f.dataType, (DoubleType, FloatType))
]

n_inf_total = 0
for c in numeric_cols:
    n_inf = df.filter(
        F.col(c).isin([float("inf"), float("-inf")]) | F.isnan(F.col(c))
    ).count()
    if n_inf > 0:
        n_inf_total += n_inf
        print(f"  ⚠️ {c}: {n_inf:,} valores inf/NaN")
        df = df.withColumn(
            c,
            F.when(
                F.isnan(F.col(c)) |
                (F.col(c) == float("inf")) |
                (F.col(c) == float("-inf")),
                F.lit(None)
            ).otherwise(F.col(c))
        )

rows_before = df.count()
df = df.dropna(subset=numeric_cols)
rows_after = df.count()

print(f"\n  Total inf/NaN detectados: {n_inf_total:,}")
print(f"  Filas eliminadas:     	{rows_before - rows_after:,}")
print(f"  Filas restantes:      	{rows_after:,}")

🔍 Reemplazando infinitos y NaN por null...


  ⚠️ Flow Byts/s: 94,296 valores inf/NaN


  ⚠️ Flow Pkts/s: 94,296 valores inf/NaN


[Stage 198:==================================================>  (121 + 7) / 128]


  Total inf/NaN detectados: 188,592
  Filas eliminadas:     	94,296
  Filas restantes:      	15,705,452


---
## 3.4 — Filtrado de Timestamps Inválidos (año < 2000)

**Evidencia:** El código temporal del NB2 detectó 14 registros con timestamps
parseados como epoch (año 1970). La columna `source_file` ya no existe
(eliminada en NB1), por lo que no es posible recuperar la fecha real.
Al ser solo 14 registros, se descartan sin impacto.

In [6]:
print("🔍 Filtrando timestamps inválidos (año < 2000)...")

if "Timestamp" in df.columns:
    df = df.withColumn(
        "_ts_parsed",
        F.to_date(F.col("Timestamp"), "dd/MM/yyyy HH:mm:ss")
    )

    n_invalid = df.filter(
        F.col("_ts_parsed").isNull() | (F.year(F.col("_ts_parsed")) < 2000)
    ).count()

    df = df.filter(
        F.col("_ts_parsed").isNotNull() & (F.year(F.col("_ts_parsed")) >= 2000)
    ).drop("_ts_parsed")

    print(f"  Timestamps inválidos eliminados: {n_invalid}")
    print(f"  Filas restantes: {df.count():,}")
else:
    print("  ⚠️ Columna 'Timestamp' no encontrada — se omite este paso")

🔍 Filtrando timestamps inválidos (año < 2000)...


  Timestamps inválidos eliminados: 14


[Stage 210:====================================================>(127 + 1) / 128]

  Filas restantes: 15,705,438


---
## 3.5 — Clipeo de Valores Negativos

**Evidencia:** NB1 (describe) mostró `Flow Duration` min = −919.011.000.000.
Las features del CSE-CIC-IDS 2018 representan conteos, tamaños y duraciones (≥ 0).
Los valores negativos son artefactos de CICFlowMeter.

> **Ref:** ml-ids (cstub) reemplaza negativos por NaN **excepto** en
> `Init Fwd Win Byts` y `Init Bwd Win Byts` (usan −1 como centinela de "no disponible").
> Göcs & Johanyák (2023) eliminan filas con negativos directamente.

**Estrategia adoptada:** Clipear a 0 en todas las columnas numéricas.
Para `Init Fwd/Bwd Win Byts`, el −1 se convierte en 0 (valor neutro para ML,
preferible al centinela −1 que confundiría al algoritmo).

In [7]:
print("🔍 Clipeando valores negativos a 0...")

numeric_cols = [
    f.name for f in df.schema.fields
    if isinstance(f.dataType, (DoubleType, FloatType, IntegerType, LongType))
]

neg_cols = []
for c in numeric_cols:
    n_neg = df.filter(F.col(c) < 0).count()
    if n_neg > 0:
        neg_cols.append(c)
        df = df.withColumn(c, F.when(F.col(c) < 0, F.lit(0)).otherwise(F.col(c)))
        print(f"  ⚠️ {c}: {n_neg:,} valores negativos → clipeados a 0")

if not neg_cols:
    print("  ✅ No se encontraron valores negativos.")
else:
    print(f"\n  Columnas afectadas: {len(neg_cols)}")

🔍 Clipeando valores negativos a 0...


  ⚠️ Flow IAT Min: 1 valores negativos → clipeados a 0


  ⚠️ Fwd IAT Min: 1 valores negativos → clipeados a 0


  ⚠️ Init Fwd Win Byts: 4,429,169 valores negativos → clipeados a 0


  ⚠️ Init Bwd Win Byts: 8,077,288 valores negativos → clipeados a 0


[Stage 457:================================>                    (77 + 47) / 124]


  Columnas afectadas: 4


---
## 3.6 — Eliminación de Columnas con Varianza Cero (post-limpieza)

NB1 verificó varianza > 0 antes de limpiar, pero tras eliminar infinitos,
duplicados y negativos, alguna columna podría haberse vuelto constante.

In [8]:
print("🔍 Detectando columnas con varianza cero...")

numeric_cols = [
    f.name for f in df.schema.fields
    if isinstance(f.dataType, (DoubleType, FloatType, IntegerType, LongType))
]

min_exprs = [F.min(F.col(c)).alias(f"min__{c}") for c in numeric_cols]
max_exprs = [F.max(F.col(c)).alias(f"max__{c}") for c in numeric_cols]
stats = df.select(min_exprs + max_exprs).collect()[0]

zero_var_cols = []
for c in numeric_cols:
    v_min = stats[f"min__{c}"]
    v_max = stats[f"max__{c}"]
    if v_min is not None and v_max is not None and v_min == v_max:
        zero_var_cols.append(c)

if zero_var_cols:
    df = df.drop(*zero_var_cols)
    print(f"  ⚠️ Eliminadas {len(zero_var_cols)} columnas constantes: {zero_var_cols}")
else:
    print("  ✅ Todas las columnas tienen varianza > 0")

print(f"  Columnas restantes: {len(df.columns)}")

🔍 Detectando columnas con varianza cero...


[Stage 460:====================================================>(123 + 1) / 124]

  ⚠️ Eliminadas 8 columnas constantes: ['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Byts/b Avg', 'Fwd Pkts/b Avg', 'Fwd Blk Rate Avg', 'Bwd Byts/b Avg', 'Bwd Pkts/b Avg', 'Bwd Blk Rate Avg']
  Columnas restantes: 72


---
## 3.7 — Validación de Integridad

Verificación final: 0 nulos, 0 infinitos, 0 negativos, 0 identificadores.

In [9]:
print("🔍 Validación de integridad post-limpieza...")
print("=" * 60)

numeric_cols_final = [
    f.name for f in df.schema.fields
    if isinstance(f.dataType, (DoubleType, FloatType, IntegerType, LongType))
]

# Nulos en numéricas
null_total = df.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in numeric_cols_final]
).collect()[0]
has_nulls = any(null_total[c] > 0 for c in numeric_cols_final)
print(f"  ¿Nulos en numéricas?    	{'⚠️ SÍ' if has_nulls else '✅ No'}")

# Infinitos
inf_count = 0
for c in [f.name for f in df.schema.fields if isinstance(f.dataType, (DoubleType, FloatType))]:
    inf_count += df.filter(
        (F.col(c) == float("inf")) | (F.col(c) == float("-inf"))
    ).count()
print(f"  ¿Infinitos residuales?  	{'⚠️ SÍ' if inf_count > 0 else '✅ No'}")

# Negativos
neg_count = 0
for c in numeric_cols_final:
    neg_count += df.filter(F.col(c) < 0).count()
print(f"  ¿Negativos residuales?  	{'⚠️ SÍ' if neg_count > 0 else '✅ No'}")

# Identificadores
id_residual = [c for c in ["Flow ID", "Src IP", "Dst IP", "Src Port"]
            if c in df.columns]
print(f"  ¿Identificadores residuales? {'⚠️ SÍ: ' + str(id_residual) if id_residual else '✅ No'}")

# Timestamp conservado
print(f"  ¿Timestamp conservado?   	{'✅ Sí (para NB4)' if 'Timestamp' in df.columns else '⚠️ No'}")

# Distribución de clases
print("\n─── Distribución de clases tras limpieza ───\n")
current_rows = df.count()
class_dist = (df.groupBy("Label")
            .count()
            .withColumn("pct", F.round(F.col("count") / current_rows * 100, 2))
            .orderBy(F.desc("count")))
class_dist.show(truncate=False)

print("=" * 60)

🔍 Validación de integridad post-limpieza...


  ¿Nulos en numéricas?    	✅ No


  ¿Infinitos residuales?  	✅ No


  ¿Negativos residuales?  	✅ No
  ¿Identificadores residuales? ✅ No
  ¿Timestamp conservado?   	✅ Sí (para NB4)

─── Distribución de clases tras limpieza ───



[Stage 822:===================================================> (124 + 4) / 128]

+------------------------+--------+-----+
|Label                   |count   |pct  |
+------------------------+--------+-----+
|Benign                  |13352487|85.02|
|DDoS attack-HOIC        |668461  |4.26 |
|DDoS attacks-LOIC-HTTP  |576175  |3.67 |
|DoS attacks-Hulk        |434873  |2.77 |
|Bot                     |282310  |1.8  |
|Infilteration           |160604  |1.02 |
|SSH-Bruteforce          |117322  |0.75 |
|DoS attacks-GoldenEye   |41455   |0.26 |
|FTP-BruteForce          |39346   |0.25 |
|DoS attacks-SlowHTTPTest|19462   |0.12 |
|DoS attacks-Slowloris   |10285   |0.07 |
|DDoS attack-LOIC-UDP    |1730    |0.01 |
|Brute Force -Web        |611     |0.0  |
|Brute Force -XSS        |230     |0.0  |
|SQL Injection           |87      |0.0  |
+------------------------+--------+-----+



---
## 3.8 — Guardado Final

Se guarda el dataset limpio con todas las features individuales,
la columna `Label` (string) y `Timestamp` (string).

**NO se aplican** en este notebook:
- Label encoding → NB5 (pre-entrenamiento)
- VectorAssembler → NB5 (tras feature engineering)
- StandardScaler → NB5 (fit solo en train para evitar data leakage)

Esto permite que NB4 trabaje con features individuales legibles
y pueda añadir/eliminar columnas libremente.

In [10]:
output_path = os.path.join(DATA_CLEAN_PATH, "preprocessed")

print(f"💾 Guardando en {output_path}...")
print(f"  Columnas: {len(df.columns)}")

df.write.mode("overwrite").parquet(output_path)

# Verificación
df_check = spark.read.parquet(output_path)
final_rows = df_check.count()
final_cols = len(df_check.columns)

from pathlib import Path

print("\n" + "=" * 70)
print("📋 RESUMEN FINAL — NB3 LIMPIEZA")
print("=" * 70)
print(f"  Filas iniciales: 	{initial_rows:,}")
print(f"  Filas finales:   	{final_rows:,}")
print(f"  Filas eliminadas:	{initial_rows - final_rows:,} ({(initial_rows - final_rows) / initial_rows * 100:.2f}%)")
print(f"  Columnas iniciales:  {initial_cols}")
print(f"  Columnas finales:	{final_cols}")
print(f"  Columnas eliminadas: {initial_cols - final_cols}")
print(f"  Timestamp:       	{'✅ Conservado' if 'Timestamp' in df_check.columns else '❌ Eliminado'}")
if os.path.exists(output_path):
    total_size = sum(f.stat().st_size for f in Path(output_path).rglob("*") if f.is_file())
    print(f"  Tamaño en disco: 	{total_size / (1024**3):.2f} GB")
print(f"  Ruta salida:     	{output_path}")
print("=" * 70)
print("\n➡️ Siguiente: NB4 — Feature Engineering y Selección de Variables")
print("   · Timestamp disponible para extraer features temporales (hour, day_of_week)")
print("   · Escalado y encoding se harán en NB5 tras train/test split")

💾 Guardando en /home/hpc/22231088student/tfg_ids/data/clean/preprocessed...
  Columnas: 72



📋 RESUMEN FINAL — NB3 LIMPIEZA
  Filas iniciales: 	16,232,943
  Filas finales:   	15,705,438
  Filas eliminadas:	527,505 (3.25%)
  Columnas iniciales:  84
  Columnas finales:	72
  Columnas eliminadas: 12
  Timestamp:       	✅ Conservado
  Tamaño en disco: 	2.10 GB
  Ruta salida:     	/home/hpc/22231088student/tfg_ids/data/clean/preprocessed

➡️ Siguiente: NB4 — Feature Engineering y Selección de Variables
   · Timestamp disponible para extraer features temporales (hour, day_of_week)
   · Escalado y encoding se harán en NB5 tras train/test split


In [12]:
spark.stop()